# Spectral Analysis Tools

pysndlib exposes sndlib's FFT, analysis windows and polar conversion -- enough to build spectra and spectrograms of your own sounds.

Source: [CLM -- FFT-related functions](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import os, importlib
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm

SR = 44100

## FFT windows

`make_fft_window(kind, size)` builds the standard windows. sidelobe suppression vs main-lobe width is the tradeoff:

In [ ]:
n = 512
windows = [('rectangular', clm.make_fft_window(clm.Window.RECTANGULAR, n)),
           ('hamming', clm.make_fft_window(clm.Window.HAMMING, n)),
           ('hann', clm.make_fft_window(clm.Window.HANN, n)),
           ('blackman2', clm.make_fft_window(clm.Window.BLACKMAN2, n))]
plt.figure(figsize=(14, 3))
for label, w in windows:
    w = np.asarray(w) / max(w)
    plt.plot(w, label=label, lw=0.9)
plt.legend(); plt.title('analysis windows'); plt.show()

## mus_fft and polar form

`mus_fft` transforms in place (real, imag); `mus_rectangular2polar` converts to magnitude/phase:

In [ ]:
x = np.sin(2 * np.pi * np.arange(1024) * 440 / SR) * np.hanning(1024)
rl = np.copy(x); im = np.zeros(1024)
clm.mus_fft(rl, im, 1024, 1)
clm.mus_rectangular2polar(rl, im)

plt.figure(figsize=(14, 3))
plt.plot(rl[:512], lw=0.8)
plt.title('a 440 Hz sine: magnitude spectrum'); plt.show()
print('peak bin:', np.argmax(rl[:512]), '-> Hz:', np.argmax(rl[:512]) * SR / 1024)

## A spectrum of a rendered sound

In [ ]:
with clm.Sound('spectral_demo.wav', channels=1, srate=SR):
    car = clm.make_oscil(440.0); mod = clm.make_oscil(440.0)
    idx = clm.make_env([0, 0, 100, 4], duration=1.0)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=1.0)
    for i in range(SR):
        clm.outa(i, clm.env(amp) * clm.oscil(car, clm.oscil(mod) * clm.env(idx) * clm.hz2radians(440.0)))

In [ ]:
d, _ = clm.file2ndarray('spectral_demo.wav')

def spectrum(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

f = np.arange(2048) * SR / 4096
plt.figure(figsize=(14, 3))
plt.plot(f, spectrum(d[0][int(SR*0.5):int(SR*0.5)+4096]), lw=0.8)
plt.xlim(0, 5000); plt.title('FM tone: carrier and sidebands'); plt.show()

In [ ]:
ipd.Audio('spectral_demo.wav', rate=SR)

## Windowing matters

An FFT over an arbitrary cut of a signal acts as if multiplied by a rectangular window -- energy smears. Compare the same signal with and without a Hann window:

In [ ]:
x = d[0][int(SR*0.5):int(SR*0.5)+4096]
bare = spectrum(x)
windowed = spectrum(x * np.hanning(4096))

plt.figure(figsize=(14, 3))
plt.semilogy(f, bare + 1e-9, lw=0.8, label='no window')
plt.semilogy(f, windowed + 1e-9, lw=0.8, label='Hann window')
plt.xlim(0, 5000); plt.ylim(1e-4, 1e2); plt.legend()
plt.title(' sidelobes: windowed vs raw'); plt.show()

## Summary

- `make_fft_window`, `mus_fft`, `mus_rectangular2polar` are a complete spectral toolkit
- Always window before FFT-ing a slice

Next: [sound files and formats](15_Sound_Files.ipynb)